## Convert C code to JAX

This section translates the provided C code into JAX, leveraging JAX's array-based operations and functional programming principles for efficient computation and automatic differentiation capabilities.

In [7]:
import jax
import jax.numpy as jnp

# JAX equivalent of norm_pts function
def norm_pts_jax(pt1_coords, pt2_coords):
    """Calculates the squared Euclidean distance between two points."""
    return jnp.sum((pt1_coords - pt2_coords)**2, axis=-1)

In [8]:
def compute_matrices_jax(nb_pts, sigma, max_val, pts_coords, connections_indices):
    """
    Computes affinity and similarity matrices, and updates point affinities.

    Args:
        nb_pts (int): Number of points.
        sigma (float): Sigma parameter for similarity calculation.
        max_val (float): Maximum value for normalization.
        pts_coords (jnp.array): (nb_pts, 3) array of point coordinates (x, y, z).
        connections_indices (jnp.array): (num_connections, 2) array of (idx1, idx2) pairs.

    Returns:
        jnp.array: Total affinity.
        jnp.array: A (normalized distance) matrix.
        jnp.array: S (similarity) matrix.
    """
    # Compute all pairwise squared distances
    diff = pts_coords[:, None, :] - pts_coords[None, :, :]  # Shape (nb_pts, nb_pts, 3)
    dist_sq_matrix = jnp.sum(diff**2, axis=-1)  # Shape (nb_pts, nb_pts)

    # Normalize distances (A matrix)
    A = dist_sq_matrix / (max_val * max_val)

    # Compute similarity matrix (S)
    S = jnp.exp(-A / sigma)

    # Initialize affinities and neighbor counts for points
    # In C, pts[i].aff starts at 1, pts[i].nb_neighbours starts at 0
    initial_affinities = jnp.ones(nb_pts, dtype=jnp.float32)

    # Function to update affinities and neighbor counts for a single connection
    def update_for_connection(carry, connection_pair):
        current_affinities, current_nb_neighbours = carry
        idx1, idx2 = connection_pair

        similarity_val = S[idx1, idx2]

        # Update affinity for idx1
        current_affinities = current_affinities.at[idx1].multiply(similarity_val)
        # Update affinity for idx2
        current_affinities = current_affinities.at[idx2].multiply(similarity_val)

        # Increment neighbor count for idx1
        current_nb_neighbours = current_nb_neighbours.at[idx1].add(1)
        # Increment neighbor count for idx2
        current_nb_neighbours = current_nb_neighbours.at[idx2].add(1)

        return (current_affinities, current_nb_neighbours), None

    # Use jax.lax.scan to iterate over connections and update state functionally
    # We pass the initial affinities and zeros for nb_neighbours
    (final_affinities, final_nb_neighbours), _ = jax.lax.scan(
        update_for_connection,
        (initial_affinities, jnp.zeros(nb_pts, dtype=jnp.int32)),
        connections_indices
    )

    # Return the total affinity (sum of squared final affinities)
    total_affinity = jnp.sum(final_affinities**2)

    # Note: In the C code, connection->d is updated. We'll just return the values here.
    # Also, pts[idx].nb_neighbours are updated. We return the final_nb_neighbours if needed.
    return total_affinity, A, S

In [9]:
# --- JAX equivalent of main function logic ---

# Parameters
nb_pts = 5
sigma = 20.0
max_val = 10.0

# Point coordinates (equivalent to pts[nb_pts] in C, excluding aff and nb_neighbours)
pts_coords = jnp.array([
    [1.0, 1.0, 1.0],  # pt[0]
    [0.0, 1.0, 1.0],  # pt[1]
    [0.0, 0.0, 1.0],  # pt[2]
    [-1.0, -1.0, 1.0], # pt[3]
    [-1.0, -1.0, -1.0] # pt[4]
], dtype=jnp.float32)

# Prepare connections (converting linked list to a JAX array of indices)
connections = []
for i in range(nb_pts):
    for j in range(i + 1, nb_pts, i + 1 if i > 0 else 1): # i+1 if i>0 else 1 to avoid infinite loop when i=0
        connections.append([i, j])

connections_indices = jnp.array(connections, dtype=jnp.int32)

print(f"JAX: sigma = {sigma} \t -- \t max_val = {max_val}")

print("JAX: Looking at connections:")
for conn_idx in connections_indices:
    print(f"JAX: Connection ({conn_idx[0]}, {conn_idx[1]}) ")

# Call the JAX main driver
total_affinity_jax, A_jax, S_jax = compute_matrices_jax(nb_pts, sigma, max_val, pts_coords, connections_indices)

print(f"JAX: Total affinity = {total_affinity_jax}")

JAX: sigma = 20.0 	 -- 	 max_val = 10.0
JAX: Looking at connections:
JAX: Connection (0, 1) 
JAX: Connection (0, 2) 
JAX: Connection (0, 3) 
JAX: Connection (0, 4) 
JAX: Connection (1, 2) 
JAX: Connection (1, 4) 
JAX: Connection (2, 3) 
JAX: Connection (3, 4) 
JAX: Total affinity = 4.922741889953613


### Explanation of JAX Conversion:

1.  **Data Representation**: C structs like `pt` and `connection` are converted to JAX NumPy arrays (`jnp.array`).
    *   `pts` (points) become `pts_coords`, a `(nb_pts, 3)` array for `(x, y, z)` coordinates.
    *   `aff` and `nb_neighbours` are managed as separate arrays or within `jax.lax.scan` for functional updates.
    *   The `connection` linked list is transformed into a `(num_connections, 2)` array `connections_indices`, where each row represents `(idx1, idx2)`.

2.  **`norm_pts_jax`**: This directly translates the squared Euclidean distance calculation using JAX array operations, leveraging broadcasting.

3.  **`compute_matrices_jax`**:
    *   **Pairwise Distances (`A`) and Similarities (`S`)**: The nested loops for calculating `A` and `S` in C are replaced by highly optimized JAX array operations. The `diff = pts_coords[:, None, :] - pts_coords[None, :, :]` creates a `(nb_pts, nb_pts, 3)` array of all pairwise differences, which is then squared and summed along the last axis to get the `dist_sq_matrix`.
    *   **Connection Updates**: The imperative loop over the linked list of connections in C, which mutates `pts[idx].aff` and `pts[idx].nb_neighbours`, is handled functionally in JAX using `jax.lax.scan`.
        *   `jax.lax.scan` is a JAX primitive for writing efficient, functional loops. It takes a `carry` (state that evolves across iterations) and an `input` (elements iterated over).
        *   The `update_for_connection` function defines how `affinities` and `nb_neighbours` are updated for each `connection_pair`.
        *   `array.at[idx].multiply(value)` and `array.at[idx].add(value)` are used for functional, in-place-like updates to arrays in JAX, which generate new arrays without mutation.
    *   The `total_affinity` is then computed as the sum of the squared final affinities.

4.  **`main` logic**: The setup of `nb_pts`, `sigma`, `max_val`, and the initial point coordinates are directly translated. The connections are constructed into a `jnp.array` before being passed to `compute_matrices_jax`.